In [3]:
# ============================================================
# LEVIR-CD STANDALONE CHANGE DETECTION
# ============================================================
#
# PIPELINE:
#
# Original 1024x1024 images
#          ↓
#      crop once
#          ↓
# Cached 256x256 tiles
#          ↓
# Siamese U-Net
#          ↓
# Binary change mask
#
# 100 EPOCHS
# ============================================================

import os
import gc
import time
import shutil
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1. PATHS
# ============================================================

CSV_PATH = (
    r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET"
    r"\final_dataset.csv"
)

BASE_DIR = (
    r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET"
)

CACHE_DIR = os.path.join(
    BASE_DIR,
    "CACHED_256_TILES"
)

TRAIN_CACHE = os.path.join(
    CACHE_DIR,
    "train"
)

VAL_CACHE = os.path.join(
    CACHE_DIR,
    "val"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "CHANGE_DETECTION_MODEL"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 2. TRAINING CONFIG
# ============================================================

EPOCHS = 100

BATCH_SIZE = 8

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-5

TILE_SIZE = 256

# Already calculated from the complete training set
POS_WEIGHT_VALUE = 18.459623377272464

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# 3. RANDOM SEED
# ============================================================

SEED = 42

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 4. GPU INFORMATION
# ============================================================

print("=" * 75)
print("LEVIR-CD STANDALONE CHANGE DETECTION")
print("=" * 75)

print(
    "\nDevice:",
    DEVICE
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )


# ============================================================
# 5. LOAD FINAL DATASET
# ============================================================

print("\nLoading final dataset...")

df = pd.read_csv(
    CSV_PATH
)

train_df = (
    df[
        df["split"] == "train"
    ]
    .reset_index(drop=True)
)

val_df = (
    df[
        df["split"] == "val"
    ]
    .reset_index(drop=True)
)

print(
    "Total samples:",
    len(df)
)

print(
    "Train samples:",
    len(train_df)
)

print(
    "Validation samples:",
    len(val_df)
)


# ============================================================
# 6. CACHE PREPROCESSING FUNCTION
# ============================================================

def prepare_cache(
    dataframe,
    split_name,
    cache_root
):

    split_dir = os.path.join(
        cache_root,
        split_name
    )

    os.makedirs(
        split_dir,
        exist_ok=True
    )

    a_dir = os.path.join(
        split_dir,
        "A"
    )

    b_dir = os.path.join(
        split_dir,
        "B"
    )

    mask_dir = os.path.join(
        split_dir,
        "mask"
    )

    os.makedirs(
        a_dir,
        exist_ok=True
    )

    os.makedirs(
        b_dir,
        exist_ok=True
    )

    os.makedirs(
        mask_dir,
        exist_ok=True
    )

    total = len(dataframe)

    print("\n" + "-" * 75)
    print(
        f"Preparing {split_name} cache: "
        f"{total} tiles"
    )
    print("-" * 75)

    start_time = time.time()

    for i, row in dataframe.iterrows():

        filename = str(
            row["filename"]
        )

        # Remove extension
        tile_name = os.path.splitext(
            filename
        )[0]

        a_output = os.path.join(
            a_dir,
            tile_name + ".png"
        )

        b_output = os.path.join(
            b_dir,
            tile_name + ".png"
        )

        mask_output = os.path.join(
            mask_dir,
            tile_name + ".png"
        )

        # ----------------------------------------------------
        # If tile already exists, skip it
        # ----------------------------------------------------

        if (
            os.path.exists(a_output)
            and
            os.path.exists(b_output)
            and
            os.path.exists(mask_output)
        ):

            if (
                (i + 1) % 500 == 0
                or
                i == 0
                or
                i == total - 1
            ):

                print(
                    f"[{i+1}/{total}] "
                    "already cached"
                )

            continue

        # ----------------------------------------------------
        # Read original 1024x1024 images
        # ----------------------------------------------------

        image_a = Image.open(
            row["cd_A"]
        ).convert("RGB")

        image_b = Image.open(
            row["cd_B"]
        ).convert("RGB")

        mask = Image.open(
            row["mask"]
        ).convert("L")

        # ----------------------------------------------------
        # Tile position
        # ----------------------------------------------------

        tile_row = int(
            row["row"]
        )

        tile_col = int(
            row["col"]
        )

        left = (
            tile_col *
            TILE_SIZE
        )

        top = (
            tile_row *
            TILE_SIZE
        )

        right = (
            left +
            TILE_SIZE
        )

        bottom = (
            top +
            TILE_SIZE
        )

        # ----------------------------------------------------
        # Crop exact corresponding tile
        # ----------------------------------------------------

        image_a = image_a.crop(
            (
                left,
                top,
                right,
                bottom
            )
        )

        image_b = image_b.crop(
            (
                left,
                top,
                right,
                bottom
            )
        )

        mask = mask.crop(
            (
                left,
                top,
                right,
                bottom
            )
        )

        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        image_a.save(
            a_output
        )

        image_b.save(
            b_output
        )

        mask.save(
            mask_output
        )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            (i + 1) % 100 == 0
            or
            i == 0
            or
            i == total - 1
        ):

            elapsed = (
                time.time()
                -
                start_time
            )

            rate = (
                (i + 1)
                /
                max(elapsed, 0.001)
            )

            remaining = (
                total - (i + 1)
            )

            eta = (
                remaining /
                max(rate, 0.001)
            )

            print(
                f"[{i+1:5d}/{total}] "
                f"{rate:.1f} tiles/s | "
                f"ETA {eta:.1f}s"
            )

    elapsed = (
        time.time()
        -
        start_time
    )

    print(
        f"\n{split_name} cache complete "
        f"in {elapsed/60:.2f} minutes."
    )


# ============================================================
# 7. CREATE CACHED TILES
# ============================================================

print("\n")
print("=" * 75)
print("STEP 1 — CACHING 256x256 TILES")
print("=" * 75)

prepare_cache(
    train_df,
    "train",
    CACHE_DIR
)

prepare_cache(
    val_df,
    "val",
    CACHE_DIR
)


# ============================================================
# 8. VERIFY CACHE
# ============================================================

print("\n")
print("=" * 75)
print("STEP 2 — VERIFYING CACHED TILES")
print("=" * 75)


def verify_tile(
    dataframe,
    split_name,
    cache_root,
    index
):

    row = dataframe.iloc[index]

    filename = str(
        row["filename"]
    )

    tile_name = os.path.splitext(
        filename
    )[0]

    a_path = os.path.join(
        cache_root,
        split_name,
        "A",
        tile_name + ".png"
    )

    b_path = os.path.join(
        cache_root,
        split_name,
        "B",
        tile_name + ".png"
    )

    mask_path = os.path.join(
        cache_root,
        split_name,
        "mask",
        tile_name + ".png"
    )

    a = np.asarray(
        Image.open(a_path)
        .convert("RGB"),
        dtype=np.float32
    ) / 255.0

    b = np.asarray(
        Image.open(b_path)
        .convert("RGB"),
        dtype=np.float32
    ) / 255.0

    mask = np.asarray(
        Image.open(mask_path)
        .convert("L"),
        dtype=np.float32
    ) / 255.0

    print(
        f"\nSample {index}"
    )

    print(
        "Filename:",
        filename
    )

    print(
        "Tile:",
        int(row["tile"]),
        "Row:",
        int(row["row"]),
        "Col:",
        int(row["col"])
    )

    print(
        "A shape:",
        a.shape
    )

    print(
        "B shape:",
        b.shape
    )

    print(
        "Mask shape:",
        mask.shape
    )

    print(
        "Mask change:",
        f"{mask.mean()*100:.4f}%"
    )

    print(
        "A-B difference:",
        f"{np.mean(np.abs(a-b)):.6f}"
    )


for i in range(
    min(5, len(train_df))
):

    verify_tile(
        train_df,
        "train",
        CACHE_DIR,
        i
    )


# ============================================================
# 9. CACHED DATASET
# ============================================================

class CachedChangeDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        split_name,
        cache_root
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )

        self.split_name = split_name

        self.cache_root = cache_root

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        filename = str(
            row["filename"]
        )

        tile_name = os.path.splitext(
            filename
        )[0]

        base = os.path.join(
            self.cache_root,
            self.split_name
        )

        a_path = os.path.join(
            base,
            "A",
            tile_name + ".png"
        )

        b_path = os.path.join(
            base,
            "B",
            tile_name + ".png"
        )

        mask_path = os.path.join(
            base,
            "mask",
            tile_name + ".png"
        )

        # ----------------------------------------------------
        # Read cached 256x256 tiles
        # ----------------------------------------------------

        image_a = np.asarray(
            Image.open(a_path)
            .convert("RGB"),
            dtype=np.float32
        ) / 255.0

        image_b = np.asarray(
            Image.open(b_path)
            .convert("RGB"),
            dtype=np.float32
        ) / 255.0

        mask = np.asarray(
            Image.open(mask_path)
            .convert("L"),
            dtype=np.float32
        ) / 255.0

        # ----------------------------------------------------
        # HWC -> CHW
        # ----------------------------------------------------

        image_a = torch.from_numpy(
            image_a
        ).permute(
            2,
            0,
            1
        )

        image_b = torch.from_numpy(
            image_b
        ).permute(
            2,
            0,
            1
        )

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        mask = (
            mask > 0.5
        ).float()

        return (
            image_a,
            image_b,
            mask
        )


# ============================================================
# 10. DATASETS
# ============================================================

print("\n")
print("=" * 75)
print("STEP 3 — CREATING DATASETS")
print("=" * 75)

train_dataset = CachedChangeDataset(
    train_df,
    "train",
    CACHE_DIR
)

val_dataset = CachedChangeDataset(
    val_df,
    "val",
    CACHE_DIR
)

print(
    "Train:",
    len(train_dataset)
)

print(
    "Val:",
    len(val_dataset)
)


# ============================================================
# 11. DATA LOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Val batches:",
    len(val_loader)
)


# ============================================================
# 12. TEST ONE BATCH
# ============================================================

print("\n")
print("=" * 75)
print("STEP 4 — TESTING DATA LOADER")
print("=" * 75)

loader_start = time.time()

test_batch = next(
    iter(train_loader)
)

test_a, test_b, test_mask = (
    test_batch
)

print(
    "Batch A:",
    tuple(test_a.shape)
)

print(
    "Batch B:",
    tuple(test_b.shape)
)

print(
    "Batch mask:",
    tuple(test_mask.shape)
)

print(
    "Mask unique values:",
    torch.unique(
        test_mask
    ).tolist()
)

print(
    "Batch loading time:",
    f"{time.time()-loader_start:.2f}s"
)


# ============================================================
# 13. MODEL
# ============================================================

print("\n")
print("=" * 75)
print("STEP 5 — CREATING SIAMESE U-NET")
print("=" * 75)


class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )
        )

    def forward(self, x):

        return self.block(x)


class SiameseUNet(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # Shared encoder
        # ----------------------------------------------------

        self.enc1 = DoubleConv(
            3,
            32
        )

        self.enc2 = DoubleConv(
            32,
            64
        )

        self.enc3 = DoubleConv(
            64,
            128
        )

        self.enc4 = DoubleConv(
            128,
            256
        )

        self.pool = nn.MaxPool2d(
            2
        )

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        # ----------------------------------------------------
        # Output
        # ----------------------------------------------------

        self.mask_head = nn.Conv2d(
            32,
            1,
            1
        )

    def encode(self, x):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        e4 = self.enc4(
            self.pool(e3)
        )

        return e1, e2, e3, e4

    def forward(
        self,
        image_a,
        image_b
    ):

        a1, a2, a3, a4 = self.encode(
            image_a
        )

        b1, b2, b3, b4 = self.encode(
            image_b
        )

        # Feature differences

        d1 = torch.abs(
            a1 - b1
        )

        d2 = torch.abs(
            a2 - b2
        )

        d3 = torch.abs(
            a3 - b3
        )

        d4 = torch.abs(
            a4 - b4
        )

        # Decoder

        x = self.up3(d4)

        x = torch.cat(
            [x, d3],
            dim=1
        )

        x = self.dec3(x)

        x = self.up2(x)

        x = torch.cat(
            [x, d2],
            dim=1
        )

        x = self.dec2(x)

        x = self.up1(x)

        x = torch.cat(
            [x, d1],
            dim=1
        )

        x = self.dec1(x)

        return self.mask_head(x)


model = SiameseUNet().to(
    DEVICE
)

print(
    "Model parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)


# ============================================================
# 14. FORWARD PASS TEST
# ============================================================

print("\nTesting forward pass...")

test_a = test_a.to(
    DEVICE
)

test_b = test_b.to(
    DEVICE
)

with torch.no_grad():

    test_output = model(
        test_a,
        test_b
    )

print(
    "Output shape:",
    tuple(test_output.shape)
)

assert test_output.shape == (
    BATCH_SIZE,
    1,
    256,
    256
)

print(
    "Forward pass: OK"
)

del test_a
del test_b
del test_output
del test_batch

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# 15. LOSS
# ============================================================

POS_WEIGHT = torch.tensor(
    [POS_WEIGHT_VALUE],
    dtype=torch.float32,
    device=DEVICE
)

bce_loss = nn.BCEWithLogitsLoss(
    pos_weight=POS_WEIGHT
)


def dice_loss(
    logits,
    targets,
    smooth=1.0
):

    probabilities = torch.sigmoid(
        logits
    )

    probabilities = probabilities.reshape(
        probabilities.size(0),
        -1
    )

    targets = targets.reshape(
        targets.size(0),
        -1
    )

    intersection = (
        probabilities *
        targets
    ).sum(dim=1)

    denominator = (
        probabilities.sum(dim=1)
        +
        targets.sum(dim=1)
    )

    dice = (
        2.0 * intersection
        +
        smooth
    ) / (
        denominator
        +
        smooth
    )

    return (
        1.0 -
        dice.mean()
    )


def combined_loss(
    logits,
    targets
):

    bce = bce_loss(
        logits,
        targets
    )

    dice = dice_loss(
        logits,
        targets
    )

    return (
        bce + dice,
        bce,
        dice
    )


# ============================================================
# 16. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 17. SCHEDULER
# ============================================================

scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=7,
        min_lr=1e-6
    )
)


# ============================================================
# 18. AMP
# ============================================================

USE_AMP = torch.cuda.is_available()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


# ============================================================
# 19. OUTPUT FILES
# ============================================================

BEST_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "best_change_detection_model.pth"
)

LAST_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "last_change_detection_model.pth"
)

HISTORY_PATH = os.path.join(
    OUTPUT_DIR,
    "change_detection_training_history.csv"
)


# ============================================================
# 20. TRAINING
# ============================================================

print("\n")
print("=" * 75)
print("STEP 6 — STARTING 100 EPOCH TRAINING")
print("=" * 75)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "AMP:",
    USE_AMP
)

print(
    "Starting now..."
)

history = []

best_iou = -1.0

best_epoch = 0

training_start = time.time()


for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = time.time()

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    train_total = 0.0
    train_bce = 0.0
    train_dice = 0.0

    for batch_idx, (
        image_a,
        image_b,
        masks
    ) in enumerate(
        train_loader
    ):

        image_a = image_a.to(
            DEVICE,
            non_blocking=True
        )

        image_b = image_b.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            logits = model(
                image_a,
                image_b
            )

            loss, bce, dice = (
                combined_loss(
                    logits,
                    masks
                )
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        train_total += (
            loss.item()
        )

        train_bce += (
            bce.item()
        )

        train_dice += (
            dice.item()
        )

        # ----------------------------------------------------
        # Progress every 200 batches
        # ----------------------------------------------------

        if (
            batch_idx == 0
            or
            (batch_idx + 1) % 200 == 0
            or
            batch_idx == len(train_loader) - 1
        ):

            print(
                f"\rEpoch {epoch:03d} | "
                f"Batch {batch_idx+1:4d}/"
                f"{len(train_loader)} | "
                f"Loss {loss.item():.4f}",
                end="",
                flush=True
            )

    print()

    train_total /= len(
        train_loader
    )

    train_bce /= len(
        train_loader
    )

    train_dice /= len(
        train_loader
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_total = 0.0
    val_bce = 0.0
    val_dice_loss = 0.0

    intersection = 0
    union = 0

    predicted_pixels = 0
    target_pixels = 0

    true_positive = 0
    false_positive = 0
    false_negative = 0

    correct_pixels = 0
    total_pixels = 0

    probability_sum = 0.0
    probability_count = 0

    with torch.no_grad():

        for (
            image_a,
            image_b,
            masks
        ) in val_loader:

            image_a = image_a.to(
                DEVICE,
                non_blocking=True
            )

            image_b = image_b.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                logits = model(
                    image_a,
                    image_b
                )

                loss, bce, dice = (
                    combined_loss(
                        logits,
                        masks
                    )
                )

            val_total += (
                loss.item()
            )

            val_bce += (
                bce.item()
            )

            val_dice_loss += (
                dice.item()
            )

            probabilities = torch.sigmoid(
                logits
            )

            predictions = (
                probabilities >= 0.5
            )

            targets = (
                masks >= 0.5
            )

            intersection += (
                (
                    predictions &
                    targets
                )
                .sum()
                .item()
            )

            union += (
                (
                    predictions |
                    targets
                )
                .sum()
                .item()
            )

            predicted_pixels += (
                predictions
                .sum()
                .item()
            )

            target_pixels += (
                targets
                .sum()
                .item()
            )

            true_positive += (
                (
                    predictions &
                    targets
                )
                .sum()
                .item()
            )

            false_positive += (
                (
                    predictions &
                    ~targets
                )
                .sum()
                .item()
            )

            false_negative += (
                (
                    ~predictions &
                    targets
                )
                .sum()
                .item()
            )

            correct_pixels += (
                (
                    predictions ==
                    targets
                )
                .sum()
                .item()
            )

            total_pixels += (
                targets.numel()
            )

            probability_sum += (
                probabilities
                .sum()
                .item()
            )

            probability_count += (
                probabilities.numel()
            )


    # ========================================================
    # VALIDATION METRICS
    # ========================================================

    val_total /= len(
        val_loader
    )

    val_bce /= len(
        val_loader
    )

    val_dice_loss /= len(
        val_loader
    )

    val_iou = (
        intersection /
        max(union, 1)
    )

    val_dice = (
        2 * intersection /
        max(
            predicted_pixels +
            target_pixels,
            1
        )
    )

    val_precision = (
        true_positive /
        max(
            true_positive +
            false_positive,
            1
        )
    )

    val_recall = (
        true_positive /
        max(
            true_positive +
            false_negative,
            1
        )
    )

    val_pixel_accuracy = (
        correct_pixels /
        total_pixels
    )

    predicted_change_ratio = (
        predicted_pixels /
        total_pixels
    )

    target_change_ratio = (
        target_pixels /
        total_pixels
    )

    mean_probability = (
        probability_sum /
        probability_count
    )


    # ========================================================
    # LEARNING RATE
    # ========================================================

    scheduler.step(
        val_iou
    )

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )


    # ========================================================
    # SAVE BEST
    # ========================================================

    is_best = (
        val_iou > best_iou
    )

    if is_best:

        best_iou = val_iou

        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "best_iou":
                    best_iou,

                "pos_weight":
                    POS_WEIGHT_VALUE
            },
            BEST_MODEL_PATH
        )


    # ========================================================
    # HISTORY
    # ========================================================

    epoch_time = (
        time.time()
        -
        epoch_start
    )

    history.append(
        {
            "epoch": epoch,

            "train_loss":
                train_total,

            "train_bce":
                train_bce,

            "train_dice_loss":
                train_dice,

            "val_loss":
                val_total,

            "val_bce":
                val_bce,

            "val_dice_loss":
                val_dice_loss,

            "val_iou":
                val_iou,

            "val_dice":
                val_dice,

            "val_precision":
                val_precision,

            "val_recall":
                val_recall,

            "val_pixel_accuracy":
                val_pixel_accuracy,

            "predicted_change_percent":
                predicted_change_ratio * 100,

            "target_change_percent":
                target_change_ratio * 100,

            "mean_probability":
                mean_probability,

            "learning_rate":
                current_lr,

            "epoch_seconds":
                epoch_time
        }
    )

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )


    # ========================================================
    # PRINT
    # ========================================================

    best_marker = (
        "  <-- BEST"
        if is_best
        else ""
    )

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"Train {train_total:.4f} | "
        f"Val {val_total:.4f} | "
        f"IoU {val_iou:.4f} | "
        f"Dice {val_dice:.4f} | "
        f"Precision {val_precision:.4f} | "
        f"Recall {val_recall:.4f} | "
        f"Pred {predicted_change_ratio*100:.3f}% | "
        f"GT {target_change_ratio*100:.3f}% | "
        f"MeanProb {mean_probability:.4f} | "
        f"LR {current_lr:.2e} | "
        f"{epoch_time:.1f}s"
        f"{best_marker}"
    )

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 21. SAVE LAST MODEL
# ============================================================

torch.save(
    {
        "epoch": EPOCHS,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "best_iou":
            best_iou,

        "best_epoch":
            best_epoch,

        "pos_weight":
            POS_WEIGHT_VALUE
    },
    LAST_MODEL_PATH
)


# ============================================================
# 22. FINAL RESULTS
# ============================================================

total_minutes = (
    time.time()
    -
    training_start
) / 60

print("\n")
print("=" * 75)
print("TRAINING COMPLETE")
print("=" * 75)

print(
    f"Best validation IoU: "
    f"{best_iou:.4f}"
)

print(
    f"Best epoch: "
    f"{best_epoch}"
)

print(
    f"Training time: "
    f"{total_minutes:.2f} minutes"
)

print(
    "\nBest model:"
)

print(
    BEST_MODEL_PATH
)

print(
    "\nLast model:"
)

print(
    LAST_MODEL_PATH
)

print(
    "\nHistory:"
)

print(
    HISTORY_PATH
)

print(
    "\nCached tiles:"
)

print(
    CACHE_DIR
)

print(
    "\nDONE."
)

LEVIR-CD STANDALONE CHANGE DETECTION

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA: 12.8

Loading final dataset...
Total samples: 7264
Train samples: 6345
Validation samples: 919


STEP 1 — CACHING 256x256 TILES

---------------------------------------------------------------------------
Preparing train cache: 6345 tiles
---------------------------------------------------------------------------
[    1/6345] 11.3 tiles/s | ETA 562.9s
[  100/6345] 15.4 tiles/s | ETA 405.2s
[  200/6345] 15.8 tiles/s | ETA 388.6s
[  300/6345] 15.6 tiles/s | ETA 387.0s
[  400/6345] 15.6 tiles/s | ETA 380.0s
[  500/6345] 15.3 tiles/s | ETA 383.2s
[  600/6345] 15.4 tiles/s | ETA 373.5s
[  700/6345] 15.6 tiles/s | ETA 361.6s
[  800/6345] 15.5 tiles/s | ETA 356.6s
[  900/6345] 15.9 tiles/s | ETA 343.5s
[ 1000/6345] 15.8 tiles/s | ETA 338.3s
[ 1100/6345] 15.9 tiles/s | ETA 330.6s
[ 1200/6345] 15.8 tiles/s | ETA 325.7s
[ 1300/6345] 15.7 tiles/s | ETA 320.5s
[ 1400/6345] 15.9 tiles/s | ETA 310.7s
[ 1